# rho review notebook

This notebook is for browsing saved outputs only. It does not rerun the unfolding.

Recommended split:
- Use `unfolder_v4_rho.ipynb` only to produce outputs under `outputs/rho`.
- Use this notebook to inspect them.
- If you want fast scrolling outside Jupyter, run `python3 outputs/build_rho_gallery.py` and open `outputs/rho/index.html`.

In [1]:
from collections import defaultdict
from pathlib import Path
import subprocess
import sys

from IPython.display import HTML, Image, Markdown, clear_output, display

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

def find_output_root():
    candidates = [
        Path.cwd() / 'outputs' / 'rho',
        Path.cwd() / '../outputs/rho',
        Path.cwd() / 'unfold' / 'outputs' / 'rho',
    ]
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved.exists():
            return resolved
    return candidates[0].resolve()


OUTPUT_ROOT = find_output_root()
PREVIEW_ROOT = OUTPUT_ROOT / '_previews'
GALLERY_SCRIPT = OUTPUT_ROOT.parent / 'build_rho_gallery.py'
ALLOWED_SUFFIXES = {'.pdf', '.png', '.jpg', '.jpeg'}
FOLDER_ORDER = ['.', 'unfold', 'uncertainties', 'data_mc']


def collect_files(root=OUTPUT_ROOT):
    files = []
    for path in sorted(root.rglob('*')):
        if not path.is_file():
            continue
        if '_previews' in path.parts:
            continue
        if path.suffix.lower() not in ALLOWED_SUFFIXES:
            continue
        files.append(path)
    return files


def folder_name(path):
    rel = path.parent.relative_to(OUTPUT_ROOT).as_posix()
    return rel or '.'


def summarize(files):
    counts = defaultdict(int)
    for path in files:
        counts[folder_name(path)] += 1
    ordered = []
    for folder in FOLDER_ORDER:
        if folder in counts:
            ordered.append((folder, counts.pop(folder)))
    ordered.extend(sorted(counts.items()))
    rows = ['| folder | files |', '|---|---:|']
    rows.extend(f'| `{folder}` | {count} |' for folder, count in ordered)
    return Markdown('\n'.join(rows))


def preview_path_for(path):
    return (PREVIEW_ROOT / path.relative_to(OUTPUT_ROOT)).with_suffix('.png')


def ensure_previews():
    if not GALLERY_SCRIPT.exists():
        raise FileNotFoundError(f'Missing gallery builder: {GALLERY_SCRIPT}')
    subprocess.run(
        [sys.executable, str(GALLERY_SCRIPT), '--root', str(OUTPUT_ROOT)],
        check=True,
        cwd=str(OUTPUT_ROOT.parent.parent),
    )


def ensure_pdf_preview(path):
    preview = preview_path_for(path)
    if preview.exists() and preview.stat().st_mtime >= path.stat().st_mtime:
        return preview
    ensure_previews()
    return preview


FILES = collect_files()
display(Markdown(f'Loaded **{len(FILES)}** saved files from `{OUTPUT_ROOT}`.'))
display(summarize(FILES))

Loaded **216** saved files from `/mnt/8A04C21E04C20CDF/wsLinux/unfold/outputs/rho`.

| folder | files |
|---|---:|
| `.` | 18 |
| `unfold` | 46 |
| `uncertainties` | 56 |
| `data_mc` | 96 |

In [2]:
if widgets is None:
    display(Markdown('`ipywidgets` is not available in this kernel. Use the next cell for a plain file list, or use `outputs/rho/index.html`.'))
else:
    folder_options = ['all'] + FOLDER_ORDER + sorted({folder_name(path) for path in FILES if folder_name(path) not in FOLDER_ORDER})
    folder = widgets.Dropdown(options=folder_options, value='all', description='folder')
    suffix = widgets.Dropdown(options=['all', '.pdf', '.png', '.jpg', '.jpeg'], value='all', description='type')
    search = widgets.Text(value='', placeholder='filename contains...', description='match')
    chooser = widgets.Select(options=[], rows=18, description='file', layout=widgets.Layout(width='100%'))
    out = widgets.Output()
    meta = widgets.HTML()
    prev_button = widgets.Button(description='Prev')
    next_button = widgets.Button(description='Next')
    rebuild_button = widgets.Button(description='Build previews')

    def filtered_files():
        items = []
        term = search.value.strip().lower()
        for path in FILES:
            current_folder = folder_name(path)
            if folder.value != 'all' and current_folder != folder.value:
                continue
            if suffix.value != 'all' and path.suffix.lower() != suffix.value:
                continue
            haystack = f'{current_folder}/{path.name}'.lower()
            if term and term not in haystack:
                continue
            items.append(path)
        return items

    def refresh_list(*_):
        items = filtered_files()
        chooser.options = [(path.relative_to(OUTPUT_ROOT).as_posix(), str(path)) for path in items]
        meta.value = f'<pre style="margin:0">{len(items)} visible files</pre>'
        if items:
            chooser.value = str(items[0])
        else:
            chooser.value = None
            with out:
                clear_output()
                display(Markdown('No files match the current filters.'))

    def render(*_):
        with out:
            clear_output()
            if not chooser.value:
                display(Markdown('Select a file to preview.'))
                return
            path = Path(chooser.value)
            rel = path.relative_to(OUTPUT_ROOT).as_posix()
            display(Markdown(f'### `{rel}`'))
            display(HTML(f'<a href="{path.as_uri()}" target="_blank">open file in browser tab</a>'))
            if path.suffix.lower() == '.pdf':
                preview = ensure_pdf_preview(path)
                if preview.exists():
                    display(Image(filename=str(preview), width=1100))
                else:
                    display(Markdown(f'No preview available for `{rel}`.'))
            else:
                display(Image(filename=str(path), width=1100))

    def rebuild_previews(_=None):
        with out:
            clear_output()
            display(Markdown('Rebuilding cached PDF previews...'))
        ensure_previews()
        render()

    def step(delta):
        options = list(chooser.options)
        if not options or chooser.value is None:
            return
        values = [value for _, value in options]
        index = values.index(chooser.value)
        chooser.value = values[(index + delta) % len(values)]

    prev_button.on_click(lambda _: step(-1))
    next_button.on_click(lambda _: step(1))
    rebuild_button.on_click(rebuild_previews)

    folder.observe(refresh_list, names='value')
    suffix.observe(refresh_list, names='value')
    search.observe(refresh_list, names='value')
    chooser.observe(render, names='value')

    controls = widgets.VBox([
        widgets.HBox([folder, suffix]),
        search,
        widgets.HBox([prev_button, next_button, rebuild_button, meta]),
        chooser,
    ])
    display(widgets.HBox([controls], layout=widgets.Layout(width='100%')))
    display(out)
    refresh_list()

Output()

In [3]:
plain_rows = [path.relative_to(OUTPUT_ROOT).as_posix() for path in FILES]
display(Markdown('## Plain file list'))
display(HTML('<pre style="max-height: 28rem; overflow-y: auto; border: 1px solid #ddd; padding: 1rem;">' + '\n'.join(plain_rows) + '</pre>'))

## Plain file list